# 06_E1_Summary — Tổng hợp kết quả baseline / E1

Notebook này tự động:

1. Đọc `metrics.json` của **LightGCN** cho 3 seed: 2023, 2024, 2025.
2. Đọc `metrics.json` của **RLMRec-Con (`lightgcn_plus`)** cho 3 seed.
3. Tìm kết quả **Most Popular**.
4. Tạo bảng raw theo từng seed và bảng cuối `mean ± SD`.
5. Lưu CSV/JSON/Markdown lên Drive.

> Most Popular chỉ chạy 1 lần vì deterministic.  
> LightGCN và RLMRec-Con được tổng hợp trên 3 seed.


In [ ]:
# CELL 1 — Mount Drive + cấu hình

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path("/content/drive/MyDrive/Data_SNA_Group14")
SEEDS = [2023, 2024, 2025]

MODELS = {
    "LightGCN": "lightgcn",
    "RLMRec-Con": "lightgcn_plus",
}

# SD trên toàn bộ các run đã thực hiện.
# Nếu muốn sample SD thì đổi thành 1.
STD_DDOF = 0

OUT_DIR = ROOT / "experiments" / "E1" / "summary"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("OUT :", OUT_DIR)


In [ ]:
# CELL 2 — Hàm đọc metric

TARGET_METRICS = ["Recall@10", "Recall@20", "NDCG@10", "NDCG@20"]

def normalize_key(s):
    return str(s).strip().lower().replace(" ", "").replace("_", "").replace("-", "")

TARGET_MAP = {normalize_key(k): k for k in TARGET_METRICS}

def flatten_dict(d, prefix=""):
    out = {}
    if isinstance(d, dict):
        for k, v in d.items():
            key = f"{prefix}.{k}" if prefix else str(k)
            if isinstance(v, dict):
                out.update(flatten_dict(v, key))
            else:
                out[key] = v
    return out

def extract_metrics(payload):
    flat = flatten_dict(payload)
    found = {}

    for full_key, value in flat.items():
        last = full_key.split(".")[-1]
        nk = normalize_key(last)
        if nk in TARGET_MAP:
            try:
                found[TARGET_MAP[nk]] = float(value)
            except Exception:
                pass

    for target_norm, target_name in TARGET_MAP.items():
        if target_name in found:
            continue
        for full_key, value in flat.items():
            if target_norm in normalize_key(full_key):
                try:
                    found[target_name] = float(value)
                    break
                except Exception:
                    pass

    return found

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def find_most_popular_metrics(root):
    candidates = [
        root / "experiments" / "E2" / "most_popular" / "metrics.json",
        root / "experiments" / "E1" / "most_popular" / "metrics.json",
        root / "experiments" / "most_popular" / "metrics.json",
        root / "experiments" / "predictions" / "most_popular" / "metrics.json",
    ]

    for p in candidates:
        if p.exists():
            return p

    exp_root = root / "experiments"
    if exp_root.exists():
        for p in exp_root.rglob("metrics.json"):
            s = str(p).lower()
            if "most" in s and "popular" in s:
                return p

    return None


In [ ]:
# CELL 3 — Đọc 6 run của LightGCN và RLMRec-Con

rows = []
missing = []

for display_name, folder_name in MODELS.items():
    for seed in SEEDS:
        path = ROOT / "experiments" / "E1" / folder_name / f"seed_{seed}" / "metrics.json"

        if not path.exists():
            missing.append(str(path))
            continue

        payload = read_json(path)
        metrics = extract_metrics(payload)

        row = {
            "Method": display_name,
            "Seed": seed,
            "metrics_path": str(path),
        }

        for m in TARGET_METRICS:
            row[m] = metrics.get(m, np.nan)

        rows.append(row)

raw_df = pd.DataFrame(rows)

print("=== RAW E1 METRICS ===")
if len(raw_df):
    display(raw_df[["Method", "Seed"] + TARGET_METRICS])
else:
    display(raw_df)

if missing:
    print("\n⚠️ Thiếu metrics.json:")
    for p in missing:
        print(" -", p)
else:
    print("\n✅ Đã tìm thấy đủ 6 metrics.json.")


In [ ]:
# CELL 4 — Đọc Most Popular

mp_path = find_most_popular_metrics(ROOT)
mp_row = None

if mp_path is None:
    print("⚠️ Chưa tìm thấy metrics.json của Most Popular.")
else:
    payload = read_json(mp_path)
    metrics = extract_metrics(payload)

    mp_row = {
        "Method": "Most Popular",
        "Seed": None,
        "metrics_path": str(mp_path),
    }

    for m in TARGET_METRICS:
        mp_row[m] = metrics.get(m, np.nan)

    print("✅ Most Popular metrics:", mp_path)
    display(pd.DataFrame([mp_row])[["Method", "Seed"] + TARGET_METRICS])


In [ ]:
# CELL 5 — Kiểm tra completeness

issues = []

required_count = len(MODELS) * len(SEEDS)

if len(raw_df) != required_count:
    issues.append("Chưa đủ 6 run LightGCN/RLMRec-Con.")

for method in MODELS.keys():
    sub = raw_df[raw_df["Method"] == method] if len(raw_df) else pd.DataFrame()
    seeds_found = sorted(sub["Seed"].dropna().astype(int).tolist()) if len(sub) else []

    if seeds_found != SEEDS:
        issues.append(f"{method}: hiện có seeds {seeds_found}, cần {SEEDS}.")

    if len(sub):
        for metric in TARGET_METRICS:
            if metric not in sub.columns or sub[metric].isna().any():
                issues.append(f"{method}: thiếu giá trị {metric}.")

if mp_row is None:
    issues.append("Chưa tìm thấy Most Popular metrics.")
else:
    for metric in TARGET_METRICS:
        if pd.isna(mp_row.get(metric)):
            issues.append(f"Most Popular: thiếu {metric}.")

if issues:
    print("⚠️ BASELINE SUMMARY CHƯA COMPLETE:")
    for x in issues:
        print(" -", x)
else:
    print("✅ BASELINE METRICS COMPLETE")


In [ ]:
# CELL 6 — Tính mean ± SD và tạo bảng cuối

summary_rows = []
numeric_rows = []

# Most Popular
if mp_row is not None:
    row = {"Method": "Most Popular"}
    for metric in TARGET_METRICS:
        val = mp_row.get(metric, np.nan)
        row[metric] = f"{val:.6f}" if pd.notna(val) else "MISSING"
    summary_rows.append(row)

# LightGCN / RLMRec-Con
for method in MODELS.keys():
    sub = raw_df[raw_df["Method"] == method] if len(raw_df) else pd.DataFrame()

    fmt_row = {"Method": method}
    num_row = {"Method": method}

    for metric in TARGET_METRICS:
        vals = sub[metric].dropna().astype(float).to_numpy() if len(sub) and metric in sub.columns else np.array([])

        if len(vals) == 0:
            mean = np.nan
            sd = np.nan
            fmt_row[metric] = "MISSING"
        else:
            mean = float(np.mean(vals))
            sd = float(np.std(vals, ddof=STD_DDOF))
            fmt_row[metric] = f"{mean:.6f} ± {sd:.6f}"

        num_row[f"{metric}_mean"] = mean
        num_row[f"{metric}_sd"] = sd

    summary_rows.append(fmt_row)
    numeric_rows.append(num_row)

summary_df = pd.DataFrame(summary_rows)
numeric_summary_df = pd.DataFrame(numeric_rows)

print("=== E1 / BASELINE SUMMARY ===")
display(summary_df)


In [ ]:
# CELL 7 — Bảng raw từng seed

raw_display_rows = []

if mp_row is not None:
    raw_display_rows.append({
        "Method": "Most Popular",
        "Seed": "-",
        **{m: mp_row.get(m, np.nan) for m in TARGET_METRICS}
    })

if len(raw_df):
    for _, r in raw_df.sort_values(["Method", "Seed"]).iterrows():
        raw_display_rows.append({
            "Method": r["Method"],
            "Seed": int(r["Seed"]),
            **{m: r[m] for m in TARGET_METRICS}
        })

raw_display_df = pd.DataFrame(raw_display_rows)
display(raw_display_df)


In [ ]:
# CELL 8 — Lưu kết quả lên Drive

raw_csv = OUT_DIR / "raw_seed_metrics.csv"
summary_csv = OUT_DIR / "e1_baseline_summary.csv"
numeric_csv = OUT_DIR / "e1_numeric_mean_sd.csv"
summary_json = OUT_DIR / "e1_baseline_summary.json"

raw_display_df.to_csv(raw_csv, index=False)
summary_df.to_csv(summary_csv, index=False)
numeric_summary_df.to_csv(numeric_csv, index=False)

json_payload = {
    "seeds": SEEDS,
    "std_ddof": STD_DDOF,
    "raw_runs": raw_display_df.to_dict(orient="records"),
    "formatted_summary": summary_df.to_dict(orient="records"),
    "numeric_summary": numeric_summary_df.to_dict(orient="records"),
    "most_popular_metrics_path": str(mp_path) if mp_path else None,
}

with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(json_payload, f, indent=2, ensure_ascii=False)

print("✅ Đã lưu:")
for p in [raw_csv, summary_csv, numeric_csv, summary_json]:
    print(" -", p)


In [ ]:
# CELL 9 — Trạng thái cuối

if not issues:
    print("✅ E1 / BASELINE SUMMARY: COMPLETE")
    print("Đã có bảng:")
    print(" - Most Popular")
    print(" - LightGCN: mean ± SD trên 3 seeds")
    print(" - RLMRec-Con: mean ± SD trên 3 seeds")
    print("\nBước tiếp theo: E2 — Popularity Analysis trên Top-K đã export.")
else:
    print("⚠️ E1 / BASELINE SUMMARY: INCOMPLETE")
    print("Xử lý các mục thiếu ở CELL 5 rồi chạy lại CELL 3 → CELL 9.")
